# Trustworthy AI — Week 7
## Robustness of AI Models

**A hands-on companion notebook to the lecture.**

This notebook turns the W7 slides into small, runnable experiments. Each section states the
idea from the slide, then demonstrates it with a *tiny, self-contained* model so everything
runs in a few seconds on a free Colab GPU.

> **How to use:** `Runtime → Run all`, or run section by section. The **Setup** cells must
> run first; after them the three parts are largely independent.

---

### Roadmap (mirrors the lecture outline)

**Part A — Robustness Metrics**
1. Accuracy Under Perturbations
2. Adversarial Robustness (ASR & robust accuracy)
3. Stability Metrics (Lipschitz / sensitivity)
4. Calibration & Confidence (ECE)
5. Robustness to Distribution Shift (ID → OOD)
6. Composite Robustness Score

**Part B — Certified Accuracy**
7. Randomized Smoothing (certified radius)
8. Convex Relaxation (Interval Bound Propagation)
9. Trade-offs: certified robustness vs. clean accuracy

**Part C — Evaluation Methods**
10. Empirical Adversarial Testing (FGSM, PGD)
11. Stress Testing with Natural Perturbations (ImageNet-C style)
12. Out-of-Distribution Evaluation (MNIST → Fashion-MNIST)
13. Certified Evaluation
14. A Mini RobustBench Leaderboard
15. Human-in-the-Loop Evaluation (conceptual)

---
*Companion to `W7_Robustness_of_AI_models`. Educational; all models are toy models we train
ourselves so everything is fast and reproducible.*

## 0. Setup

Enable the GPU: **Runtime → Change runtime type → Hardware accelerator → GPU (T4)**.
Every tensor is placed on `DEVICE`, which is the GPU when one is available.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0); np.random.seed(0)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)
if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> GPU for the full experience.")

In [ ]:
# Shared dataset: MNIST digits. Small, fast, easy to perturb visually.
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset

_tfm = transforms.ToTensor()
_train = datasets.MNIST(root="./data", train=True,  download=True, transform=_tfm)
_test  = datasets.MNIST(root="./data", train=False, download=True, transform=_tfm)

Xtr = _train.data.float().unsqueeze(1) / 255.0      # (60000,1,28,28)
ytr = _train.targets.clone()
Xte = _test.data.float().unsqueeze(1)  / 255.0
yte = _test.targets.clone()
print("train:", Xtr.shape, " test:", Xte.shape)

### A reusable classifier + helpers

One small CNN and a `train_model` helper keep every section short. We train a base model
once here and reuse it throughout.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28->14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14->7
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(32*7*7, 64), nn.ReLU(),
                                nn.Linear(64, n_classes))
    def forward(self, x):
        return self.fc(self.conv(x))

def train_model(X, y, epochs=2, bs=256, lr=1e-3, model=None, quiet=False):
    # Train a SmallCNN on tensors X,y. Returns the model on DEVICE.
    model = (model or SmallCNN()).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X, y), batch_size=bs, shuffle=True)
    model.train()
    for ep in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            F.cross_entropy(model(xb), yb).backward()
            opt.step()
        if not quiet: print(f"  epoch {ep+1}/{epochs} done")
    return model

@torch.no_grad()
def accuracy(model, X, y, bs=1000):
    model.eval(); correct = 0
    for i in range(0, len(X), bs):
        pred = model(X[i:i+bs].to(DEVICE)).argmax(1).cpu()
        correct += (pred == y[i:i+bs]).sum().item()
    return correct / len(X)

print("Training the base model used across the notebook...")
base_model = train_model(Xtr, ytr, epochs=3)
print(f"Clean test accuracy: {accuracy(base_model, Xte, yte):.3f}")

---
# Part A — Robustness Metrics

> *"Robustness measures how well AI models maintain performance under perturbations...
> Metrics help quantify resilience against noise, shifts, and attacks."*

No single number captures robustness, so the lecture lists several complementary metrics.
We compute each one on our base model.

## 1. Accuracy Under Perturbations

**Slide idea:** the simplest robustness measure is accuracy on noisy inputs (blur, rotation,
noise). A drop shows sensitivity to input variation — a baseline but limited metric.

We sweep Gaussian-noise strength and plot the accuracy curve.

In [ ]:
sigmas = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
accs = []
for s in sigmas:
    Xnoisy = (Xte + s * torch.randn_like(Xte)).clamp(0, 1)
    accs.append(accuracy(base_model, Xnoisy, yte))
    print(f"noise sigma={s:.1f} -> accuracy={accs[-1]:.3f}")

plt.figure(figsize=(6,3))
plt.plot(sigmas, accs, "o-"); plt.xlabel("Gaussian noise σ"); plt.ylabel("accuracy")
plt.title("Accuracy under input perturbation"); plt.grid(alpha=0.3); plt.show()

# Show a few images at increasing noise so the perturbation is visible.
fig, ax = plt.subplots(1, len(sigmas), figsize=(13, 1.8))
for j, s in enumerate(sigmas):
    ax[j].imshow((Xte[0,0] + s*torch.randn_like(Xte[0,0])).clamp(0,1), cmap="gray")
    ax[j].set_title(f"σ={s}"); ax[j].axis("off")
plt.show()

## 2. Adversarial Robustness Metrics

**Slide idea:** measure performance under adversarial examples. **Attack Success Rate (ASR)**
quantifies vulnerability; **robust accuracy** is accuracy after adversarial perturbation.
These capture *worst-case* performance (vs. the average-case noise above).

We use FGSM to craft the attack and report both metrics.

In [ ]:
def fgsm(model, x, y, eps):
    # One-step adversarial perturbation of L-inf magnitude eps.
    x = x.clone().to(DEVICE).requires_grad_(True); y = y.to(DEVICE)
    F.cross_entropy(model(x), y).backward()
    return (x + eps * x.grad.sign()).clamp(0, 1).detach()

@torch.no_grad()
def _pred(model, x): return model(x.to(DEVICE)).argmax(1).cpu()

xb, yb = Xte[:2000], yte[:2000]
clean_pred = _pred(base_model, xb)
for eps in [0.05, 0.10, 0.20, 0.30]:
    x_adv = fgsm(base_model, xb, yb, eps)
    adv_pred = _pred(base_model, x_adv)
    robust_acc = (adv_pred == yb).float().mean().item()
    # ASR = fraction of originally-correct inputs that the attack flips
    correct = clean_pred == yb
    asr = (adv_pred[correct] != yb[correct]).float().mean().item()
    print(f"eps={eps:.2f} | robust accuracy={robust_acc:.3f} | attack success rate (ASR)={asr:.3f}")

## 3. Stability Metrics (Lipschitz / Sensitivity)

**Slide idea:** a stable model gives similar outputs for nearby inputs. Metrics include the
**Lipschitz constant** and sensitivity bounds; higher stability = smoother, more reliable.

We estimate a *local* Lipschitz constant empirically: perturb inputs slightly and measure
`‖f(x+δ) − f(x)‖ / ‖δ‖`. The max over many samples lower-bounds the true constant.

In [ ]:
@torch.no_grad()
def local_lipschitz(model, X, n=500, delta=1e-2, trials=5):
    model.eval()
    x = X[:n].to(DEVICE)
    base_logits = model(x)
    ratios = []
    for _ in range(trials):
        d = torch.randn_like(x); d = delta * d / d.flatten(1).norm(dim=1)[:,None,None,None]
        num = (model(x + d) - base_logits).flatten(1).norm(dim=1)
        den = d.flatten(1).norm(dim=1)
        ratios.append((num/den).cpu())
    return torch.stack(ratios).max(0).values

# Compare the base model with an adversarially-trained (smoother) model.
def train_adv(X, y, epochs=2, eps=0.2):
    m = SmallCNN().to(DEVICE); opt = torch.optim.Adam(m.parameters(), 1e-3)
    loader = DataLoader(TensorDataset(X, y), batch_size=256, shuffle=True)
    for _ in range(epochs):
        for xb_, yb_ in loader:
            xb_, yb_ = xb_.to(DEVICE), yb_.to(DEVICE)
            xa = xb_.clone().requires_grad_(True)
            F.cross_entropy(m(xa), yb_).backward()
            xa = (xa + eps*xa.grad.sign()).clamp(0,1).detach()
            opt.zero_grad(); (F.cross_entropy(m(xb_),yb_)+F.cross_entropy(m(xa),yb_)).backward(); opt.step()
    return m

print("Training a smoother (adversarially-trained) model for comparison...")
smooth_model = train_adv(Xtr, ytr)

L_base   = local_lipschitz(base_model, Xte)
L_smooth = local_lipschitz(smooth_model, Xte)
print(f"Estimated local Lipschitz  | base model  : mean={L_base.mean():.2f}  max={L_base.max():.2f}")
print(f"Estimated local Lipschitz  | smooth model: mean={L_smooth.mean():.2f}  max={L_smooth.max():.2f}")
print("Lower Lipschitz = more stable = outputs change less for nearby inputs.")

## 4. Calibration & Confidence Metrics (ECE)

**Slide idea:** robust models need *reliable confidence*. Miscalibrated models are
overconfident when wrong. **Expected Calibration Error (ECE)** measures how far confidence is
from actual accuracy.

We bin predictions by confidence and compare confidence vs. accuracy in each bin (a
reliability diagram), then compute ECE.

In [ ]:
@torch.no_grad()
def calibration(model, X, y, n_bins=10):
    model.eval()
    probs = F.softmax(model(X.to(DEVICE)), dim=1).cpu()
    conf, pred = probs.max(1)
    correct = (pred == y).float()
    bins = torch.linspace(0, 1, n_bins+1)
    ece, xs, accs, confs = 0.0, [], [], []
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i+1])
        if m.sum() == 0: continue
        bin_acc, bin_conf = correct[m].mean().item(), conf[m].mean().item()
        ece += (m.float().mean().item()) * abs(bin_acc - bin_conf)
        xs.append((bins[i]+bins[i+1]).item()/2); accs.append(bin_acc); confs.append(bin_conf)
    return ece, xs, accs, confs

ece, xs, accs, confs = calibration(base_model, Xte, yte)
print(f"Expected Calibration Error (ECE) = {ece:.3f}   (lower = better calibrated)")

plt.figure(figsize=(4,4))
plt.plot([0,1],[0,1],"--",color="gray",label="perfect calibration")
plt.plot(confs, accs, "o-", label="model")
plt.xlabel("confidence"); plt.ylabel("accuracy"); plt.title(f"Reliability diagram (ECE={ece:.3f})")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5. Robustness to Distribution Shift (ID → OOD)

**Slide idea:** measure the performance drop from in-distribution (ID) to out-of-distribution
(OOD) — domain, covariate, or concept shifts. Critical beyond training data.

We apply a *covariate shift* (rotation + contrast change) and report the ID→OOD accuracy gap.

In [ ]:
import torchvision.transforms.functional as TF

def shift_batch(X, angle=25, contrast=0.5):
    # Covariate shift: rotate and reduce contrast (a systematic domain change, not random noise).
    out = TF.rotate(X, angle)
    out = (out - 0.5) * contrast + 0.5
    return out.clamp(0, 1)

id_acc  = accuracy(base_model, Xte, yte)
Xood    = shift_batch(Xte)
ood_acc = accuracy(base_model, Xood, yte)
print(f"In-distribution (ID) accuracy : {id_acc:.3f}")
print(f"Out-of-distribution accuracy  : {ood_acc:.3f}")
print(f"ID -> OOD generalization gap  : {id_acc - ood_acc:.3f}")

fig, ax = plt.subplots(2, 6, figsize=(11, 3.4))
for j in range(6):
    ax[0,j].imshow(Xte[j,0], cmap="gray"); ax[0,j].axis("off")
    ax[1,j].imshow(Xood[j,0], cmap="gray"); ax[1,j].axis("off")
ax[0,0].set_ylabel("ID"); fig.text(0.07,0.72,"ID",rotation=90); fig.text(0.07,0.30,"OOD (shifted)",rotation=90)
plt.suptitle("Distribution shift: rotate + reduce contrast"); plt.show()

## 6. Composite Robustness Score

**Slide idea:** no single metric fully captures robustness. Composite indices combine
accuracy, stability, and calibration; suites like **RobustBench** standardize this.

We combine four normalized sub-scores into one 0–1 index and show it as a radar-style summary
for two models (base vs. adversarially-trained).

In [ ]:
def robustness_profile(model, name):
    clean = accuracy(model, Xte[:2000], yte[:2000])
    radv  = ( _pred(model, fgsm(model, Xte[:2000], yte[:2000], 0.2)) == yte[:2000] ).float().mean().item()
    lip   = local_lipschitz(model, Xte, n=300).mean().item()
    stab  = 1.0 / (1.0 + lip)                          # map Lipschitz -> [0,1], higher=better
    ece,_,_,_ = calibration(model, Xte[:2000], yte[:2000])
    cal   = 1.0 - min(ece, 1.0)                         # higher = better calibrated
    comp  = np.mean([clean, radv, stab, cal])
    print(f"{name:16s} | clean={clean:.2f} robust={radv:.2f} stability={stab:.2f} "
          f"calib={cal:.2f} | COMPOSITE={comp:.3f}")
    return dict(clean=clean, robust=radv, stability=stab, calibration=cal)

print("Composite robustness (equal-weighted mean of four sub-scores):\n")
p_base   = robustness_profile(base_model,   "base model")
p_smooth = robustness_profile(smooth_model, "adv-trained")

# radar chart
labels = list(p_base.keys())
ang = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist(); ang += ang[:1]
fig = plt.figure(figsize=(4.5,4.5)); axr = plt.subplot(111, polar=True)
for prof, nm in [(p_base,"base"),(p_smooth,"adv-trained")]:
    vals = [prof[k] for k in labels]; vals += vals[:1]
    axr.plot(ang, vals, "o-", label=nm); axr.fill(ang, vals, alpha=0.1)
axr.set_xticks(ang[:-1]); axr.set_xticklabels(labels); axr.set_ylim(0,1)
axr.set_title("Composite robustness profile"); axr.legend(loc="upper right", bbox_to_anchor=(1.3,1.1))
plt.show()

---
# Part B — Certified Accuracy

> *"Certified accuracy provides formal guarantees... it mathematically bounds model
> performance. Unlike empirical testing, certification gives provable robustness."*

Empirical robustness (Part A) is "best effort" against *known* attacks. Certification proves
that **no** perturbation within a bound can change the prediction.

## 7. Randomized Smoothing (Certified Radius)

**Slide idea:** the leading scalable certification method. Add Gaussian noise to the input,
average predictions, and derive a **provable L2 robustness radius** from the class vote
margin (Cohen et al., 2019).

A base classifier `f` becomes a smoothed classifier `g(x) = argmax_c P(f(x+ε)=c)`. If the top
class is returned with probability `p̄`, the prediction is certified within radius
`R = σ · Φ⁻¹(p̄)`.

In [ ]:
from scipy.stats import norm
from math import isnan

# Train a model WITH noise augmentation so smoothing works well (standard for smoothing).
def train_noisy(X, y, sigma=0.5, epochs=3):
    m = SmallCNN().to(DEVICE); opt = torch.optim.Adam(m.parameters(), 1e-3)
    loader = DataLoader(TensorDataset(X, y), batch_size=256, shuffle=True)
    for _ in range(epochs):
        for xb_, yb_ in loader:
            xb_, yb_ = xb_.to(DEVICE), yb_.to(DEVICE)
            xn = (xb_ + sigma*torch.randn_like(xb_)).clamp(0,1)
            opt.zero_grad(); F.cross_entropy(m(xn), yb_).backward(); opt.step()
    return m

SIGMA = 0.5
print(f"Training a noise-augmented base classifier (sigma={SIGMA})...")
g_base = train_noisy(Xtr, ytr, sigma=SIGMA)

@torch.no_grad()
def certify(model, x, sigma, n=1000, alpha=0.001, bs=500):
    # Monte-Carlo randomized smoothing certificate for a single input x (1,1,28,28).
    model.eval(); counts = torch.zeros(10)
    for i in range(0, n, bs):
        k = min(bs, n-i)
        batch = x.repeat(k,1,1,1).to(DEVICE) + sigma*torch.randn(k,1,28,28,device=DEVICE)
        preds = model(batch.clamp(0,1)).argmax(1).cpu()
        counts += torch.bincount(preds, minlength=10).float()
    top = counts.argmax().item()
    p_hat = counts[top].item()/n
    # lower-confidence bound on p (Clopper-Pearson via normal approx for simplicity)
    p_lb = p_hat - norm.ppf(1-alpha)*np.sqrt(p_hat*(1-p_hat)/n)
    radius = sigma*norm.ppf(p_lb) if p_lb > 0.5 else 0.0
    if isnan(radius): radius = 0.0
    return top, max(radius, 0.0), p_hat

# Certify a handful of test points.
print("\nPer-sample certificates (sigma=0.5):")
for i in range(6):
    cls, R, p = certify(g_base, Xte[i:i+1], SIGMA, n=1000)
    ok = "correct" if cls == yte[i].item() else "WRONG"
    print(f"  sample {i}: pred={cls} ({ok})  p̄={p:.3f}  certified L2 radius R={R:.3f}")

In [ ]:
# Certified accuracy curve: fraction of inputs correctly classified AND certified beyond radius r.
N_CERT = 150   # keep small; each sample runs many forward passes
idxs = range(N_CERT)
results = [certify(g_base, Xte[i:i+1], SIGMA, n=500) for i in idxs]
radii = np.array([R for (_, R, _) in results])
preds = np.array([c for (c, _, _) in results])
true  = yte[:N_CERT].numpy()
correct = preds == true

rs = np.linspace(0, 1.5, 40)
cert_acc = [np.mean(correct & (radii >= r)) for r in rs]

plt.figure(figsize=(6,3))
plt.plot(rs, cert_acc)
plt.xlabel("certified L2 radius r"); plt.ylabel("certified accuracy")
plt.title(f"Certified accuracy vs. radius (randomized smoothing, σ={SIGMA})")
plt.grid(alpha=0.3); plt.show()
print(f"Certified accuracy at r=0.0: {cert_acc[0]:.3f}  |  at r=0.5: {np.interp(0.5, rs, cert_acc):.3f}")

## 8. Convex Relaxation (Interval Bound Propagation)

**Slide idea:** use optimization *relaxations* to bound adversarial effects (linear
relaxation, SDP). Precise but computationally heavy — more precise than smoothing, less
scalable.

We implement the simplest linear relaxation — **Interval Bound Propagation (IBP)**. Given an
L∞ box `[x−ε, x+ε]` around the input, we propagate lower/upper bounds through each layer. If
the true-class lower bound beats every other class's upper bound, the point is *certified*.

In [ ]:
# A small fully-connected net so IBP stays readable (Conv IBP works the same, more bookkeeping).
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(784, 128); self.l2 = nn.Linear(128, 64); self.l3 = nn.Linear(64, 10)
    def forward(self, x):
        x = x.flatten(1); x = F.relu(self.l1(x)); x = F.relu(self.l2(x)); return self.l3(x)

print("Training a small MLP for IBP certification...")
mlp = MLP().to(DEVICE)
opt = torch.optim.Adam(mlp.parameters(), 1e-3)
loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=256, shuffle=True)
for _ in range(3):
    for xb_, yb_ in loader:
        xb_, yb_ = xb_.to(DEVICE), yb_.to(DEVICE)
        opt.zero_grad(); F.cross_entropy(mlp(xb_), yb_).backward(); opt.step()
print(f"MLP clean accuracy: {accuracy(mlp, Xte, yte):.3f}")

def linear_bounds(W, b, lo, hi):
    # Interval arithmetic through an affine layer y = Wx + b.
    c, r = (lo+hi)/2, (hi-lo)/2
    yc = c @ W.t() + b
    yr = r @ W.t().abs()
    return yc - yr, yc + yr

@torch.no_grad()
def ibp_certify(model, x, eps):
    # Returns True if the prediction is certified robust within L-inf ball of radius eps.
    x = x.flatten(1).to(DEVICE)
    lo, hi = (x-eps).clamp(0,1), (x+eps).clamp(0,1)
    for lin in [model.l1, model.l2]:
        lo, hi = linear_bounds(lin.weight, lin.bias, lo, hi)
        lo, hi = F.relu(lo), F.relu(hi)                 # ReLU is monotonically increasing -> bounds pass through
    lo, hi = linear_bounds(model.l3.weight, model.l3.bias, lo, hi)
    # certified if true-class lower bound > every other class's upper bound
    pred = ((lo+hi)/2).argmax(1)
    certified = torch.ones(len(x), dtype=torch.bool, device=DEVICE)
    for c in range(10):
        others = torch.arange(10, device=DEVICE) != c
        # for samples predicted c: need lo[c] > max hi[others]
        mask = pred == c
        if mask.any():
            worst_other = hi[mask][:, others].max(1).values
            certified[mask] &= lo[mask][:, c] > worst_other
    return pred.cpu(), certified.cpu()

for eps in [0.01, 0.02, 0.05, 0.1]:
    pred, cert = ibp_certify(mlp, Xte[:2000], eps)
    correct = pred == yte[:2000]
    print(f"eps={eps:.2f} | IBP certified accuracy = {(correct & cert).float().mean():.3f}")

## 9. Trade-offs in Certified Accuracy

**Slide idea:** higher certified robustness usually *lowers clean accuracy*, and stronger
guarantees cost more computation. Certification must be tailored to the application.

We train smoothed classifiers at several noise levels σ and plot clean accuracy vs. certified
accuracy — the classic robustness/accuracy trade-off.

In [ ]:
sigmas = [0.12, 0.25, 0.50, 0.75]
clean_accs, cert_accs = [], []
for s in sigmas:
    m = train_noisy(Xtr, ytr, sigma=s, epochs=2)
    clean_accs.append(accuracy(m, Xte[:2000], yte[:2000]))
    # certified accuracy at a fixed radius r=0.25
    res = [certify(m, Xte[i:i+1], s, n=300) for i in range(80)]
    r_ok = np.mean([(c==yte[i].item()) and (R>=0.25) for i,(c,R,_) in enumerate(res)])
    cert_accs.append(r_ok)
    print(f"sigma={s:.2f} | clean acc={clean_accs[-1]:.3f} | certified@r=0.25={cert_accs[-1]:.3f}")

plt.figure(figsize=(6,3))
plt.plot(sigmas, clean_accs, "o-", label="clean accuracy")
plt.plot(sigmas, cert_accs, "s-", label="certified acc @ r=0.25")
plt.xlabel("smoothing noise σ"); plt.ylabel("accuracy"); plt.legend()
plt.title("Trade-off: more noise -> more certifiable but less clean-accurate")
plt.grid(alpha=0.3); plt.show()

---
# Part C — Evaluation Methods for Model Robustness

> *"Evaluating robustness ensures models withstand real-world challenges... A mix of empirical
> and theoretical evaluations is needed."*

Part A defined metrics and Part B certified them. Part C is the *evaluation protocol*: how you
actually stress-test a model before deployment.

## 10. Empirical Adversarial Testing (FGSM & PGD)

**Slide idea:** apply attack algorithms (FGSM, PGD, CW) to stress-test models. PGD is the
standard strong first-order attack — iterated FGSM with projection back into the ε-ball.

We compare FGSM (1 step) vs. PGD (multi-step) on the base and adversarially-trained models.

In [ ]:
def pgd(model, x, y, eps=0.2, alpha=0.03, steps=10):
    # Projected Gradient Descent: iterate FGSM, projecting back into the L-inf eps-ball.
    x0 = x.clone().to(DEVICE); y = y.to(DEVICE)
    x_adv = x0 + torch.empty_like(x0).uniform_(-eps, eps)   # random start
    x_adv = x_adv.clamp(0,1).detach()
    for _ in range(steps):
        x_adv.requires_grad_(True)
        F.cross_entropy(model(x_adv), y).backward()
        with torch.no_grad():
            x_adv = x_adv + alpha * x_adv.grad.sign()
            x_adv = torch.min(torch.max(x_adv, x0-eps), x0+eps).clamp(0,1)
        x_adv = x_adv.detach()
    return x_adv

xb, yb = Xte[:2000], yte[:2000]
for name, m in [("base model", base_model), ("adv-trained", smooth_model)]:
    fa = (_pred(m, fgsm(m, xb, yb, 0.2)) == yb).float().mean().item()
    pa = (_pred(m, pgd (m, xb, yb, 0.2)) == yb).float().mean().item()
    print(f"{name:14s} | clean={accuracy(m,xb,yb):.3f} | FGSM acc={fa:.3f} | PGD acc={pa:.3f}")
print("\nPGD is a stronger attack than FGSM -> lower robust accuracy. Adv-training helps against both.")

## 11. Stress Testing with Natural Perturbations (ImageNet-C style)

**Slide idea:** test under realistic distortions — blur, rotation, brightness — as in
**ImageNet-C**. Natural robustness complements adversarial robustness.

We build a mini "MNIST-C": several corruption types at fixed severity, and report accuracy
per corruption.

In [ ]:
import torchvision.transforms.functional as TF

def corrupt(X, kind, sev=0.5):
    if kind == "gaussian_noise": return (X + sev*torch.randn_like(X)).clamp(0,1)
    if kind == "blur":           return TF.gaussian_blur(X, kernel_size=5, sigma=1+3*sev)
    if kind == "brightness":     return (X + sev).clamp(0,1)
    if kind == "contrast":       return ((X-0.5)*(1-0.8*sev)+0.5).clamp(0,1)
    if kind == "rotate":         return TF.rotate(X, int(40*sev))
    return X

kinds = ["gaussian_noise","blur","brightness","contrast","rotate"]
scores = {k: accuracy(base_model, corrupt(Xte, k), yte) for k in kinds}
scores_clean = accuracy(base_model, Xte, yte)
print(f"clean accuracy: {scores_clean:.3f}")
for k,v in scores.items(): print(f"  {k:16s}: {v:.3f}")

plt.figure(figsize=(6,3))
plt.bar(range(len(kinds)), [scores[k] for k in kinds])
plt.axhline(scores_clean, color="k", ls="--", label="clean")
plt.xticks(range(len(kinds)), kinds, rotation=30, ha="right"); plt.ylabel("accuracy")
plt.title("Accuracy under natural corruptions (mini MNIST-C)"); plt.legend(); plt.tight_layout(); plt.show()

# mean Corruption Error style summary (higher mean accuracy = more naturally robust)
print(f"\nMean corruption accuracy: {np.mean(list(scores.values())):.3f}")

## 12. Out-of-Distribution (OOD) Evaluation

**Slide idea:** evaluate on shifted distributions beyond training data (CIFAR-10 → CIFAR-10-C,
MNIST → Fashion-MNIST). A good OOD detector should be *less confident* on OOD inputs.

We feed **Fashion-MNIST** (never seen) to our MNIST model and check whether max-softmax
confidence separates in-distribution from OOD — a standard OOD-detection baseline (MSP).

In [ ]:
fashion = datasets.FashionMNIST(root="./data", train=False, download=True,
                                transform=transforms.ToTensor())
Xood = fashion.data.float().unsqueeze(1)/255.0

@torch.no_grad()
def msp(model, X, bs=1000):
    model.eval(); out=[]
    for i in range(0, len(X), bs):
        out.append(F.softmax(model(X[i:i+bs].to(DEVICE)),1).max(1).values.cpu())
    return torch.cat(out)

conf_id  = msp(base_model, Xte[:5000])
conf_ood = msp(base_model, Xood[:5000])
print(f"mean confidence  ID (MNIST)        : {conf_id.mean():.3f}")
print(f"mean confidence  OOD (Fashion-MNIST): {conf_ood.mean():.3f}")

# AUROC of the confidence-based OOD detector
from sklearn.metrics import roc_auc_score
labels = np.r_[np.ones(len(conf_id)), np.zeros(len(conf_ood))]
scores = np.r_[conf_id.numpy(), conf_ood.numpy()]
print(f"OOD-detection AUROC (max-softmax) : {roc_auc_score(labels, scores):.3f}  (1.0 = perfect)")

plt.figure(figsize=(6,3))
plt.hist(conf_id.numpy(), bins=40, alpha=0.6, label="ID (MNIST)")
plt.hist(conf_ood.numpy(), bins=40, alpha=0.6, label="OOD (Fashion-MNIST)")
plt.xlabel("max-softmax confidence"); plt.legend(); plt.title("OOD evaluation via confidence"); plt.show()

## 13. Certified Evaluation

**Slide idea:** report **certified accuracy alongside empirical accuracy** for rigorous,
mathematical evidence. We tie Parts B and C together: for the same model, show empirical
robust accuracy (PGD) next to the certified guarantee (IBP).

In [ ]:
eps = 0.05
emp = (_pred(mlp, pgd(mlp, Xte[:2000], yte[:2000], eps)) == yte[:2000]).float().mean().item()
pred, cert = ibp_certify(mlp, Xte[:2000], eps)
certified = ((pred == yte[:2000]) & cert).float().mean().item()
print(f"At L-inf eps={eps}:")
print(f"  empirical robust accuracy (PGD attack) : {emp:.3f}   <- may drop with stronger attacks")
print(f"  CERTIFIED accuracy (IBP guarantee)     : {certified:.3f}   <- holds against ANY attack in the ball")
print("\nCertified <= empirical: certification is a provable lower bound on true robustness.")

## 14. A Mini RobustBench Leaderboard

**Slide idea:** benchmark suites like **RobustBench** give a unified leaderboard with
adversarial, natural, and certified metrics for fair comparison.

We assemble our own small leaderboard across the models trained in this notebook.

In [ ]:
import pandas as pd

def evaluate_all(model):
    xb, yb = Xte[:2000], yte[:2000]
    clean = accuracy(model, xb, yb)
    pgd_a = (_pred(model, pgd(model, xb, yb, 0.2)) == yb).float().mean().item()
    nat   = np.mean([accuracy(model, corrupt(Xte[:2000], k), yte[:2000]) for k in kinds])
    ece,_,_,_ = calibration(model, xb, yb)
    return dict(clean=clean, PGD_robust=pgd_a, natural=nat, ECE=ece)

rows = {"Base (standard)": base_model,
        "Adv-trained":     smooth_model,
        "Noise-augmented": g_base}
table = pd.DataFrame({name: evaluate_all(m) for name, m in rows.items()}).T
table = table.round(3)
print(table.to_string())
print("\nRead it like RobustBench: no model wins every column -> robustness is multi-objective.")

## 15. Human-in-the-Loop Evaluation

**Slide idea:** human feedback catches failure modes automated benchmarks miss — especially
in NLP and generative models, and in *ambiguous* cases. It complements formal and empirical
evaluation.

We can't run a real human study here, but we can surface the cases a human should review:
low-confidence or borderline predictions where the model is most likely wrong. Routing these
to a human is a practical, deployable version of the idea.

In [ ]:
@torch.no_grad()
def review_queue(model, X, y, k=8):
    model.eval()
    probs = F.softmax(model(X.to(DEVICE)),1).cpu()
    conf, pred = probs.max(1)
    # "ambiguity" = gap between top-1 and top-2 probabilities (small gap = uncertain)
    top2 = probs.topk(2,1).values
    margin = (top2[:,0]-top2[:,1])
    order = margin.argsort()[:k]        # smallest margin first -> most ambiguous
    return order, pred, conf, margin

order, pred, conf, margin = review_queue(base_model, Xte[:3000], yte[:3000])
acc_on_flagged = (pred[order]==yte[order]).float().mean().item()
acc_overall    = accuracy(base_model, Xte[:3000], yte[:3000])
print(f"Model accuracy overall               : {acc_overall:.3f}")
print(f"Model accuracy on the flagged queue  : {acc_on_flagged:.3f}  <- much lower: humans add most value here")

fig, ax = plt.subplots(1, 8, figsize=(13, 1.8))
for j, idx in enumerate(order):
    ax[j].imshow(Xte[idx,0], cmap="gray"); ax[j].axis("off")
    ax[j].set_title(f"p={pred[idx].item()}/t={yte[idx].item()}\nmargin={margin[idx]:.2f}", fontsize=8)
plt.suptitle("Most ambiguous predictions -> route to a human reviewer", y=1.35); plt.show()

---
# Conclusion

| Lecture concept | Demonstrated in |
|---|---|
| Accuracy under perturbations | §1 noise-sweep curve |
| Adversarial robustness (ASR, robust acc) | §2, §10 (FGSM/PGD) |
| Stability / Lipschitz | §3 local Lipschitz estimate |
| Calibration (ECE) | §4 reliability diagram |
| Distribution shift (ID→OOD) | §5, §12 (Fashion-MNIST) |
| Composite metrics / RobustBench | §6 radar, §14 leaderboard |
| Randomized smoothing | §7 certified radius & curve |
| Convex relaxation | §8 Interval Bound Propagation |
| Robustness/accuracy trade-off | §9 σ-sweep |
| Natural corruptions (ImageNet-C) | §11 mini MNIST-C |
| Certified vs. empirical | §13 |
| Human-in-the-loop | §15 review queue |

**Key lessons, demonstrated in code:**

- **Robustness is multi-dimensional.** Average-case noise, worst-case adversarial, stability,
  calibration, and OOD each tell a different story — a model strong on one can be weak on another.
- **Empirical ≠ certified.** PGD accuracy is only a *best-effort upper estimate*; certified
  accuracy (smoothing, IBP) is a *provable lower bound* that holds against any attack in the ball.
- **Everything trades off.** More smoothing noise buys certification but costs clean accuracy;
  stronger guarantees cost compute. Certification is tailored to the application's stakes.
- **Comprehensive evaluation** — adversarial + natural + OOD + certified + human review —
  is what builds justified confidence in a model's robustness.

> *Building robust AI ensures reliability, safety, and long-term trustworthiness.*

---
### Ideas to extend
- Add the **CW attack** and compare its robust accuracy to PGD.
- Swap MNIST for **CIFAR-10** and a small ResNet to see the trade-offs sharpen.
- Implement **CROWN / α-CROWN** for tighter certificates than IBP.
- Use temperature scaling to *fix* calibration in §4 and watch ECE drop.